In [33]:
# EDINET APIからデータを取得するモジュール
import requests
import pandas as pd
import json
import time
from datetime import datetime, timedelta
import os
from dotenv import load_dotenv

In [34]:
# EDINET API のベースURL
BASE_URL = "https://disclosure.edinet-fsa.go.jp/api/v2"

In [35]:
# APIキーの読み込み
load_dotenv()
API_KEY = os.getenv("EDINET_API_KEY")
print(f"APIキー確認: {API_KEY[:5]}...")

APIキー確認: 6f3e7...


In [36]:
def get_documents_by_date(date: str) -> dict:
    url = f"{BASE_URL}/documents.json"
    params = {
        "date": date,
        "type": 2, # 有価証券報告書
        "Subscription-Key": API_KEY
    }
    response = requests.get(url, params=params)
    return response.json()

In [37]:
# テスト：直近の書類一覧を取得
test_date = "2024-01-15"
result = get_documents_by_date(test_date)
print(f"取得件数: {result.get('metadata', {}).get('resultset', {}).get('count', 0)}")
print(json.dumps(result['results'][0], ensure_ascii=False, indent=2))

取得件数: 437
{
  "seqNumber": 1,
  "docID": "S100SLX4",
  "edinetCode": "E36154",
  "secCode": null,
  "JCN": null,
  "filerName": "田畑　正吾",
  "fundCode": null,
  "ordinanceCode": "060",
  "formCode": "010002",
  "docTypeCode": "350",
  "periodStart": null,
  "periodEnd": null,
  "submitDateTime": "2024-01-15 09:00",
  "docDescription": "変更報告書",
  "issuerEdinetCode": "E34973",
  "subjectEdinetCode": null,
  "subsidiaryEdinetCode": null,
  "currentReportReason": null,
  "parentDocID": null,
  "opeDateTime": null,
  "withdrawalStatus": "0",
  "docInfoEditStatus": "0",
  "disclosureStatus": "0",
  "xbrlFlag": "1",
  "pdfFlag": "1",
  "attachDocFlag": "0",
  "englishDocFlag": "0",
  "csvFlag": "1",
  "legalStatus": "1"
}


In [44]:
# 有価証券報告書（docTypeCode: 120）のみに絞り込む関数
def filter_annual_reports(result: dict) -> list:
    """
    一般事業会社の有価証券報告書のみを抽出する
    docTypeCode: 120 = 有価証券報告書
    ordinanceCode: 010 = 一般事業会社
    """
    if not result.get('results'):
        return []
    return [
        doc for doc in result['results']
        if doc.get('docTypeCode') == '120'
        and doc.get('ordinanceCode') == '010'
    ]

In [45]:
# テスト
annual_reports = filter_annual_reports(result)
print(f"有価証券報告書の件数: {len(annual_reports)}")
if annual_reports:
    print(f"例: {annual_reports[0]['filerName']}")

有価証券報告書の件数: 1
例: 株式会社キタック


In [46]:
# 複数日付にわたって有価証券報告書を一括取得する
def get_annual_reports_bulk(start_date: str, end_date: str) -> pd.DataFrame:
    """
    指定期間の有価証券報告書一覧を取得する
    start_date, end_date: "YYYY-MM-DD" 形式
    """
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    
    all_reports = []
    current = start
    
    while current <= end:
        date_str = current.strftime("%Y-%m-%d")
        try:
            result = get_documents_by_date(date_str)
            reports = filter_annual_reports(result)
            all_reports.extend(reports)
            print(f"{date_str}: {len(reports)}件取得")
        except Exception as e:
            print(f"{date_str}: エラー {e}")
        
        time.sleep(1)  # API負荷軽減のため1秒待機
        current += timedelta(days=1)
    
    return pd.DataFrame(all_reports)

In [ ]:
# テスト：2024年6月の1週間分を取得(3月決算の会社が多いため)
df_reports = get_annual_reports_bulk("2024-06-24", "2024-06-30")
print(f"\n合計取得件数: {len(df_reports)}件")
print(df_reports[['filerName', 'docID', 'periodEnd']].head(10))

2024-06-24: 199件取得
2024-06-25: 265件取得
2024-06-26: 473件取得
2024-06-27: 703件取得
2024-06-28: 467件取得
2024-06-29: 0件取得
2024-06-30: 0件取得

合計取得件数: 2107件
          filerName     docID   periodEnd
0      日本空調サービス株式会社  S100TPNR  2024-03-31
1          今村証券株式会社  S100TNPR  2024-03-31
2        有馬冨士開発株式会社  S100TQCX  2024-03-31
3        三洋化成工業株式会社  S100TODJ  2024-03-31
4          株式会社大田花き  S100TQA4  2024-03-31
5  株式会社クリップコーポレーション  S100TQ70  2024-03-31
6          株式会社マンダム  S100TOZR  2024-03-31
7        遠州トラック株式会社  S100TPR6  2024-03-31
8          三信電気株式会社  S100TNW8  2024-03-31
9            日工株式会社  S100TO3E  2024-03-31


In [ ]:
# 業種で絞り込む（建設業を確認）
# まずどんな会社名があるか確認
print("取得企業サンプル:")
print(df_reports['filerName'].head(20).to_string())
print(f"\n総企業数: {len(df_reports)}")

# データをCSVに保存（絶対パスで指定） ※生成AI利用(エラーの連続でエラー文を"Gemini"と投げ壁打ちした結果です)
import os
save_path = os.path.join(os.path.expanduser('~'), 'quest_1', 'data', 'annual_reports_list_202406.csv')
df_reports.to_csv(save_path, index=False, encoding='utf-8-sig')
print(f"\nCSV保存完了: {save_path}")

取得企業サンプル:
0             日本空調サービス株式会社
1                 今村証券株式会社
2               有馬冨士開発株式会社
3               三洋化成工業株式会社
4                 株式会社大田花き
5         株式会社クリップコーポレーション
6                 株式会社マンダム
7               遠州トラック株式会社
8                 三信電気株式会社
9                   日工株式会社
10                丸三証券株式会社
11                日進工具株式会社
12     株式会社　昭和システムエンジニアリング
13               北海道瓦斯株式会社
14                株式会社トーモク
15                ホクシン株式会社
16    福井コンピュータホールディングス株式会社
17               株式会社ニッカトー
18                 株式会社クレオ
19                ニッコー株式会社

総企業数: 2107

CSV保存完了: /Users/takeshiokada/quest_1/data/annual_reports_list_202406.csv


In [54]:
# 日揮HDが含まれているか確認
jgc = df_reports[df_reports['filerName'].str.contains('日揮', na=False)]
print(f"日揮関連企業:")
print(jgc[['filerName', 'docID', 'periodEnd']].to_string())

日揮関連企業:
           filerName     docID   periodEnd
1500  日揮ホールディングス株式会社  S100TVFX  2024-03-31


In [56]:
# 日揮HDの有価証券報告書をダウンロード
import zipfile
import io

def download_document(doc_id: str) -> bytes:
    """
    docIDを指定して書類をZIPでダウンロードする
    """
    url = f"{BASE_URL}/documents/{doc_id}"
    params = {
        "type": 1,  # XBRLファイル含むZIP
        "Subscription-Key": API_KEY
    }
    response = requests.get(url, params=params)
    return response.content

# 日揮HDの書類をダウンロード
jgc_doc_id = "S100TVFX"
print(f"ダウンロード開始: {jgc_doc_id}")
content = download_document(jgc_doc_id)
print(f"ダウンロード完了: {len(content)}バイト")

ダウンロード開始: S100TVFX
ダウンロード完了: 1172004バイト


In [ ]:
# ZIPを展開してファイル一覧を確認
zip_file = zipfile.ZipFile(io.BytesIO(content))
file_list = zip_file.namelist()
print("ZIPの中身:")
for f in file_list:
    print(f)

ZIPの中身:
XBRL/PublicDoc/0000000_header_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0101010_honbun_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0102010_honbun_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0103010_honbun_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0104010_honbun_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0105000_honbun_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0105010_honbun_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0105020_honbun_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0105025_honbun_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0105040_honbun_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0105050_honbun_jp

In [58]:
# CSVファイルのみ抽出して確認
csv_files = [f for f in file_list if f.endswith('.csv')]
print("CSVファイル一覧:")
for f in csv_files:
    print(f)

CSVファイル一覧:


In [59]:
# HTMファイルのみ抽出して確認
htm_files = [f for f in file_list if f.endswith('.htm')]
print("HTMファイル一覧:")
for f in htm_files:
    print(f)

HTMファイル一覧:
XBRL/PublicDoc/0000000_header_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0101010_honbun_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0102010_honbun_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0103010_honbun_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0104010_honbun_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0105000_honbun_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0105010_honbun_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0105020_honbun_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0105025_honbun_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0105040_honbun_jpcrp030000-asr-001_E01575-000_2024-03-31_01_2024-06-27_ixbrl.htm
XBRL/PublicDoc/0105050_honbun

In [60]:
# 本文HTMファイルを読み込んで女性活躍指標を探す
from bs4 import BeautifulSoup

# 最初の本文ファイルを読み込む
honbun_files = [f for f in htm_files if 'honbun' in f]
print(f"本文ファイル数: {len(honbun_files)}")

# 最初のファイルを読み込む
with zip_file.open(honbun_files[0]) as f:
    soup = BeautifulSoup(f.read(), 'html.parser')
    text = soup.get_text()

# 女性活躍関連キーワードで検索
keywords = ['女性管理職', '育児休業', '賃金格差', '女性活躍']
for keyword in keywords:
    idx = text.find(keyword)
    if idx != -1:
        print(f"\n【{keyword}】")
        print(text[idx:idx+200])

本文ファイル数: 22

【育児休業】
育児休業取得率及び労働者の男女の賃金の差異











当事業年度



 

管理職に占める女性労働者の割合（％）（注）２、３、４

男性労働者の育児休業取得率（％）（注）５、６

労働者の男女の賃金の差異（％）（注）２、７



全労働者

正規雇用労働者

パート・有期労働者（注）８、９



当社

3.3

63

57.8

60.5

34.6



日揮グローバル㈱

1


In [61]:
# 女性活躍指標を数値として抽出する
def extract_diversity_metrics(zip_content: bytes, company_name: str) -> dict:
    """
    ZIPファイルから女性活躍指標を抽出する
    """
    zip_file = zipfile.ZipFile(io.BytesIO(zip_content))
    file_list = zip_file.namelist()
    honbun_files = [f for f in file_list if 'honbun' in f and f.endswith('.htm')]
    
    metrics = {
        'company_name': company_name,
        'female_manager_ratio': None,
        'male_childcare_leave_rate': None,
        'gender_wage_gap': None
    }
    
    for honbun_file in honbun_files:
        with zip_file.open(honbun_file) as f:
            soup = BeautifulSoup(f.read(), 'html.parser')
            
            tags = {
                'female_manager_ratio': ['RatioOfFemaleWorkersToCorporateOfficers',
                                        'RatioOfFemaleManagers'],
                'male_childcare_leave_rate': ['RateOfMaleWorkersOnChildcareLeave'],
                'gender_wage_gap': ['RatioOfWomenSalaryToMenSalary',
                                   'DifferenceOfWagesBetweenMaleAndFemale']
            }
            
            for metric, tag_names in tags.items():
                for tag_name in tag_names:
                    tag = soup.find(attrs={'name': lambda x: x and tag_name.lower() in x.lower()})
                    if tag and tag.get_text(strip=True):
                        try:
                            metrics[metric] = float(tag.get_text(strip=True).replace(',', ''))
                            break
                        except:
                            pass
    
    return metrics

# 日揮HDでテスト
jgc_metrics = extract_diversity_metrics(content, '日揮ホールディングス株式会社')
print("日揮HD 女性活躍指標:")
for k, v in jgc_metrics.items():
    print(f"  {k}: {v}")

日揮HD 女性活躍指標:
  company_name: 日揮ホールディングス株式会社
  female_manager_ratio: None
  male_childcare_leave_rate: None
  gender_wage_gap: None


In [ ]:
# 上記の数値が取れていない結果を踏まえて #
# XBRLタグ名を実際に確認する
with zip_file.open(honbun_files[0]) as f:
    soup = BeautifulSoup(f.read(), 'html.parser')
    
# 女性・育児・賃金に関連するタグを全部抽出
keywords = ['female', 'Female', 'childcare', 'Childcare', 'wage', 'Wage', 'salary', 'Salary']
found_tags = set()

for tag in soup.find_all(True):
    tag_name = tag.get('name', '')
    if tag_name and any(k in tag_name for k in keywords):
        found_tags.add(tag_name)

print("関連タグ一覧:")
for t in sorted(found_tags):
    print(t)

関連タグ一覧:
jpcrp_cor:AllEmployeesCalculatedBasedOnProvisionsOfArticle714Item1OfOrdinanceForEnforcementOfActOnChildcareLeaveCaregiverLeaveAndOtherMeasuresForTheWelfareOfWorkersCaringForChildrenOrOtherFamilyMembersRatioOfMaleEmployeesTakingChildcareLeaveMetricsOfReportingCompany
jpcrp_cor:AllEmployeesDifferencesInWagesBetweenMaleAndFemaleEmployeesMetricsOfConsolidatedSubsidiaries
jpcrp_cor:AllEmployeesDifferencesInWagesBetweenMaleAndFemaleEmployeesMetricsOfReportingCompany
jpcrp_cor:AllEmployeesRatioOfMaleEmployeesTakingChildcareLeaveMetricsOfConsolidatedSubsidiaries
jpcrp_cor:AverageAnnualSalaryInformationAboutReportingCompanyInformationAboutEmployees
jpcrp_cor:NonRegularEmployeesDifferencesInWagesBetweenMaleAndFemaleEmployeesMetricsOfConsolidatedSubsidiaries
jpcrp_cor:NonRegularEmployeesDifferencesInWagesBetweenMaleAndFemaleEmployeesMetricsOfReportingCompany
jpcrp_cor:RatioOfFemaleEmployeesInManagerialPositionsMetricsOfConsolidatedSubsidiaries
jpcrp_cor:RatioOfFemaleEmployeesInManagerialP

In [67]:
# 正しいタグ名で女性活躍指標を抽出する
def extract_diversity_metrics_v2(zip_content: bytes, company_name: str) -> dict:
    """
    正しいXBRLタグ名で女性活躍指標を抽出する
    """
    zip_file = zipfile.ZipFile(io.BytesIO(zip_content))
    file_list = zip_file.namelist()
    honbun_files = [f for f in file_list if 'honbun' in f and f.endswith('.htm')]
    
    metrics = {
        'company_name': company_name,
        'female_manager_ratio': None,       # 女性管理職比率
        'male_childcare_leave_rate': None,  # 男性育児休業取得率
        'gender_wage_gap': None             # 男女賃金格差（全労働者）
    }
    
    # 対応するタグ名
    tag_map = {
    'female_manager_ratio': 'jpcrp_cor:RatioOfFemaleEmployeesInManagerialPositionsMetricsOfReportingCompany',
    'male_childcare_leave_rate': 'jpcrp_cor:AllEmployeesCalculatedBasedOnProvisionsOfArticle714Item1OfOrdinanceForEnforcementOfActOnChildcareLeaveCaregiverLeaveAndOtherMeasuresForTheWelfareOfWorkersCaringForChildrenOrOtherFamilyMembersRatioOfMaleEmployeesTakingChildcareLeaveMetricsOfReportingCompany',
    'gender_wage_gap': 'jpcrp_cor:AllEmployeesDifferencesInWagesBetweenMaleAndFemaleEmployeesMetricsOfReportingCompany'
    }
    
    for honbun_file in honbun_files:
        with zip_file.open(honbun_file) as f:
            soup = BeautifulSoup(f.read(), 'html.parser')
            
            for metric, tag_name in tag_map.items():
                if metrics[metric] is not None:
                    continue
                tag = soup.find(attrs={'name': tag_name})
                if tag and tag.get_text(strip=True):
                    try:
                        metrics[metric] = float(tag.get_text(strip=True).replace(',', ''))
                    except:
                        pass
    
    return metrics

# 日揮HDでテスト
jgc_metrics = extract_diversity_metrics_v2(content, '日揮ホールディングス株式会社')
print("日揮HD 女性活躍指標:")
for k, v in jgc_metrics.items():
    print(f"  {k}: {v}")

日揮HD 女性活躍指標:
  company_name: 日揮ホールディングス株式会社
  female_manager_ratio: 3.3
  male_childcare_leave_rate: 63.0
  gender_wage_gap: 57.8


この数字の意味
・女性管理職比率3.3%は非常に低い
・男女賃金格差57.8%は女性の賃金が男性の約58%ということ
・男性育児休業取得率63%は比較的高め

In [ ]:
# ↑タグ名調査（試行錯誤） # ※生成AI利用(プロンプト：male_childcare_leave_rateがNoneがです。正しいタグ名を調べて下さい等)
# 育児休業関連のタグを全部確認する
with zip_file.open(honbun_files[0]) as f:
    soup = BeautifulSoup(f.read(), 'html.parser')

childcare_tags = {}
for tag in soup.find_all(True):
    tag_name = tag.get('name', '')
    if tag_name and 'childcare' in tag_name.lower() or (tag_name and 'childcare' in tag_name):
        childcare_tags[tag_name] = tag.get_text(strip=True)

print("育児休業関連タグと値:")
for k, v in childcare_tags.items():
    print(f"  {k}: {v}")

育児休業関連タグと値:
  jpcrp_cor:AllEmployeesCalculatedBasedOnProvisionsOfArticle714Item1OfOrdinanceForEnforcementOfActOnChildcareLeaveCaregiverLeaveAndOtherMeasuresForTheWelfareOfWorkersCaringForChildrenOrOtherFamilyMembersRatioOfMaleEmployeesTakingChildcareLeaveMetricsOfReportingCompany: 63
  jpcrp_cor:AllEmployeesRatioOfMaleEmployeesTakingChildcareLeaveMetricsOfConsolidatedSubsidiaries: 100
